In [6]:
import equinox as eqx
import jax
import jax.numpy as jnp

from pmrf.models.lines import DatasheetCoaxial

In [ ]:
coax = DatasheetCoaxial()
type(coax.length)

int

In [ ]:
class MyModule(eqx.Module):
    param_array: jax.Array
    static_float: float # This will be treated as static by default

    def __init__(self, key, initial_float_value):
        self.param_array = jax.random.normal(key, (3,))
        self.static_float = initial_float_value

    def __call__(self, x):
        return self.param_array * x + self.static_float

# Create an instance of your module
key = jax.random.PRNGKey(0)
module = MyModule(key, initial_float_value=5.0)

# JIT compile the call method using eqx.filter_jit
@eqx.filter_jit
def jitted_call(model, input_val):
    return model(input_val)

# Test with different input values for x (dynamic)
input_x1 = jnp.array([1.0, 2.0, 3.0])
output1 = jitted_call(module, input_x1)
print(f"Output 1: {output1}")

input_x2 = jnp.array([4.0, 5.0, 6.0])
output2 = jitted_call(module, input_x2)
print(f"Output 2: {output2}")

# If you change static_float, the function will recompile
module_new_static = MyModule(key, initial_float_value=10.0)
output3 = jitted_call(module_new_static, input_x1) # This will trigger a recompile
print(f"Output 3 (with new static float): {output3}")

# You can also explicitly specify how things are filtered
@eqx.filter_jit(filter_fn=eqx.is_array) # Only JAX arrays are traced
def custom_filtered_call(model, input_val):
    return model(input_val)

output4 = custom_filtered_call(module, input_x1)
print(f"Output 4 (custom filter): {output4}")